# Exploring SQL Window Functions

Window functions are one of the most powerful features in modern SQL.  
Unlike regular aggregate functions (`GROUP BY`), window functions let you **calculate values across a set of rows related to the current row** without collapsing the result into fewer rows.

### What you will learn
- The basic syntax of window functions
- `ROW_NUMBER()`, `RANK()`, `DENSE_RANK()`
- `PARTITION BY` vs no partition
- Running totals and moving averages with `SUM()` / `AVG()` OVER
- Looking at previous/next rows with `LAG()` and `LEAD()`
- Practical ranking and comparison examples using the Mubi dataset

---

## 1. Setup

In [ ]:
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

connection = sqlite3.connect("mubi_movies_ratings.db")
print("Connected to database")

---

## 2. Basic Syntax of Window Functions

```sql
function_name() OVER (
    [PARTITION BY column1, column2, ...]
    [ORDER BY column3 [ASC|DESC]]
    [ROWS / RANGE frame_specification]   -- optional, advanced
)
```

- **PARTITION BY** → divides the data into groups (like `GROUP BY`, but keeps all rows)
- **ORDER BY** → defines the order inside each partition (required for ranking and running calculations)
- The function is calculated **for each row** based on the window defined around it.

---

## 3. Ranking Functions: ROW_NUMBER, RANK, DENSE_RANK

These three functions are very similar but behave differently when there are ties.

### Example 3.1 – Rank all movies by popularity

In [ ]:
query = """
SELECT 
    movie_title,
    movie_release_year,
    movie_popularity,
    ROW_NUMBER() OVER (ORDER BY movie_popularity DESC) AS row_num,
    RANK()       OVER (ORDER BY movie_popularity DESC) AS rank_val,
    DENSE_RANK() OVER (ORDER BY movie_popularity DESC) AS dense_rank_val
FROM movies
ORDER BY movie_popularity DESC
LIMIT 15;
"""

df = pd.read_sql_query(query, connection)
df

**Key differences:**

| Function     | Behavior when there is a tie                     | Next rank after a tie |
|--------------|--------------------------------------------------|-----------------------|
| `ROW_NUMBER` | Always unique sequential numbers                 | Continues normally    |
| `RANK`       | Same rank for ties, then **skips** the next numbers | Skips                 |
| `DENSE_RANK` | Same rank for ties, **no gaps**                  | No skip               |

### Example 3.2 – Rank movies **within each decade** (PARTITION BY)

In [ ]:
query = """
SELECT 
    movie_title,
    movie_release_year,
    (movie_release_year / 10) * 10 AS decade,
    movie_popularity,
    ROW_NUMBER() OVER (
        PARTITION BY (movie_release_year / 10) * 10 
        ORDER BY movie_popularity DESC
    ) AS rank_in_decade
FROM movies
WHERE movie_release_year >= 1990
ORDER BY decade, rank_in_decade
LIMIT 30;
"""

df = pd.read_sql_query(query, connection)
df

---

## 4. Aggregate Window Functions (Running Totals & Moving Averages)

You can use `SUM()`, `AVG()`, `COUNT()`, `MIN()`, `MAX()` as window functions.

### Example 4.1 – Running total of movie popularity by release year

In [ ]:
query = """
SELECT 
    movie_release_year,
    COUNT(*) AS movies_released,
    SUM(movie_popularity) AS total_popularity,
    SUM(SUM(movie_popularity)) OVER (ORDER BY movie_release_year) AS running_total_popularity
FROM movies
WHERE movie_release_year BETWEEN 2000 AND 2021
GROUP BY movie_release_year
ORDER BY movie_release_year;
"""

df = pd.read_sql_query(query, connection)
df

### Example 4.2 – Moving average of ratings (3-year window)

In [ ]:
query = """
WITH yearly_avg AS (
    SELECT 
        movie_release_year AS year,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM movies
    WHERE movie_release_year BETWEEN 1990 AND 2021
      AND rating IS NOT NULL
    GROUP BY movie_release_year
)
SELECT 
    year,
    avg_rating,
    ROUND(AVG(avg_rating) OVER (
        ORDER BY year
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2) AS moving_avg_3yr
FROM yearly_avg
ORDER BY year;
"""

df = pd.read_sql_query(query, connection)
df

---

## 5. LAG and LEAD – Looking at Previous / Next Rows

Extremely useful for year-over-year comparisons.

In [ ]:
query = """
WITH yearly_stats AS (
    SELECT 
        movie_release_year AS year,
        COUNT(*) AS num_movies,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM movies
    WHERE movie_release_year BETWEEN 2005 AND 2021
      AND rating IS NOT NULL
    GROUP BY movie_release_year
)
SELECT 
    year,
    num_movies,
    avg_rating,
    LAG(avg_rating, 1) OVER (ORDER BY year) AS prev_year_avg,
    ROUND(avg_rating - LAG(avg_rating, 1) OVER (ORDER BY year), 2) AS change_vs_prev_year,
    LEAD(avg_rating, 1) OVER (ORDER BY year) AS next_year_avg
FROM yearly_stats
ORDER BY year;
"""

df = pd.read_sql_query(query, connection)
df

---

## 6. Practical Business Examples

### Example 6.1 – Top 3 most popular movies per decade

In [ ]:
query = """
WITH ranked AS (
    SELECT 
        movie_title,
        movie_release_year,
        (movie_release_year / 10) * 10 AS decade,
        movie_popularity,
        rating,
        ROW_NUMBER() OVER (
            PARTITION BY (movie_release_year / 10) * 10 
            ORDER BY movie_popularity DESC
        ) AS rn
    FROM movies
    WHERE movie_release_year >= 1970
)
SELECT decade, movie_title, movie_release_year, movie_popularity, rating
FROM ranked
WHERE rn <= 3
ORDER BY decade, rn;
"""

df = pd.read_sql_query(query, connection)
df

### Example 6.2 – Each movie’s popularity compared to the average of its decade

In [ ]:
query = """
SELECT 
    movie_title,
    movie_release_year,
    (movie_release_year / 10) * 10 AS decade,
    movie_popularity,
    ROUND(AVG(movie_popularity) OVER (
        PARTITION BY (movie_release_year / 10) * 10
    ), 1) AS decade_avg_popularity,
    ROUND(
        movie_popularity - AVG(movie_popularity) OVER (
            PARTITION BY (movie_release_year / 10) * 10
        ), 1
    ) AS difference_from_decade_avg
FROM movies
WHERE movie_release_year >= 2000
ORDER BY difference_from_decade_avg DESC
LIMIT 20;
"""

df = pd.read_sql_query(query, connection)
df

### Example 6.3 – Ranking directors by number of highly rated movies (using window + filter)

In [ ]:
query = """
WITH director_stats AS (
    SELECT 
        director_name,
        COUNT(*) AS total_movies,
        SUM(CASE WHEN rating >= 4.0 THEN 1 ELSE 0 END) AS high_rated_movies,
        ROUND(AVG(rating), 2) AS avg_rating
    FROM movies
    WHERE director_name IS NOT NULL
      AND rating IS NOT NULL
    GROUP BY director_name
    HAVING COUNT(*) >= 5
)
SELECT 
    director_name,
    total_movies,
    high_rated_movies,
    avg_rating,
    RANK() OVER (ORDER BY high_rated_movies DESC, avg_rating DESC) AS director_rank
FROM director_stats
ORDER BY director_rank
LIMIT 15;
"""

df = pd.read_sql_query(query, connection)
df

---

## 7. Practice Challenges

Try writing these queries yourself before looking at solutions.

### Challenge 1
Show the top 5 movies by `rating` **for each decade** starting from 1980.  
Use `ROW_NUMBER()` and `PARTITION BY`.

In [ ]:
# Your code here



### Challenge 2
For every year from 2010 onwards, show:
- Number of movies released
- Average rating that year
- The average rating of the **previous year** (`LAG`)
- The difference between current and previous year

In [ ]:
# Your code here



### Challenge 3 (Harder)
Find movies that are in the **top 10%** most popular of their release decade.  
Hint: Use `PERCENT_RANK()` or calculate rank / total count.

In [ ]:
# Your code here



---

## 8. Quick Reference – Most Used Window Functions

| Function              | Purpose                                      |
|-----------------------|----------------------------------------------|
| `ROW_NUMBER()`        | Unique sequential number                     |
| `RANK()`              | Ranking with gaps on ties                    |
| `DENSE_RANK()`        | Ranking without gaps                         |
| `NTILE(n)`            | Divide into n buckets                        |
| `LAG(col, n)`         | Value from n rows before                     |
| `LEAD(col, n)`        | Value from n rows after                      |
| `FIRST_VALUE(col)`    | First value in the window                    |
| `LAST_VALUE(col)`     | Last value in the window                     |
| `SUM() OVER()`        | Running / partitioned sum                    |
| `AVG() OVER()`        | Running / partitioned average                |
| `COUNT() OVER()`      | Running / partitioned count                  |
| `PERCENT_RANK()`      | Relative rank (0 to 1)                       |
| `CUME_DIST()`         | Cumulative distribution                      |

---

## Final Notes

- Window functions do **not** reduce the number of rows (unlike `GROUP BY`).
- Always put the `ORDER BY` inside the `OVER()` clause when ranking or calculating running totals.
- `PARTITION BY` is optional — omit it when you want to calculate across the entire result set.
- You can combine multiple window functions in the same query.
- Performance tip: Window functions can be expensive on very large tables — use them wisely.

Close the connection when you finish:

In [ ]:
connection.close()
print("Connection closed.")